[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_23_linear_pure.ipynb)

# 🟢 Easy: Linear Layer without Flax

*Core Ops & Layers*
Problem 03's linear layer, written with no Flax at all.

### Signature
```python
def init_linear(key, in_features, out_features, use_bias=True):
    ...   # -> params, a pytree of arrays

def apply_linear(params, x):
    ...   # -> (..., out_features)
```

### The parameter pytree
```python
{"kernel": (in_features, out_features),      # x @ kernel, so in comes first
 "bias":   (out_features,)}                  # omitted entirely when use_bias=False
```

`kernel` is `(in, out)` — the Flax layout, the transpose of PyTorch's. Scale it
by `1 / sqrt(in_features)`, exactly as in problem 03; `bias` starts at zeros.

**When `use_bias=False`, the `'bias'` key must not be in the dict.** A pytree
describes what exists — a present-but-zero bias would still collect gradients
and still be updated by an optimizer.

### `apply_linear` must accept any leading shape
`(B, T, in)` and `(N, in)` and a bare `(in,)` all have to work, which they do
for free if you write `x @ params["kernel"]` and never mention the batch axes.

### Why this exists alongside problem 03
Interview sandboxes (CoderPad and friends) often ship `jax` and nothing else.
Every `nnx.Module` problem in this repo is unrunnable there, and the rewrite is
not mechanical: without a module to hold state, **you** own the parameters.

The pattern below is what Flax, Haiku and Equinox all compile down to, so it is
worth being able to write from memory:

```python
params = init_x(key, ...)      # a pytree of arrays — that is the whole "layer"
y      = apply_x(params, x)    # a pure function of (params, input)
```

`jax.grad` differentiates with respect to the **first** argument, which is why
`params` goes first. Nothing else in JAX has to change: `jit`, `vmap` and
`scan` all accept a pytree of parameters as an ordinary argument.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def init_linear(key, in_features, out_features, use_bias=True):
    """Build the parameter pytree for a linear layer."""
    pass  # Replace this


def apply_linear(params, x):
    """Apply the layer: (..., in_features) -> (..., out_features)."""
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

params = init_linear(jax.random.key(0), 4, 3)
print("params:", jax.tree.map(lambda a: a.shape, params))

for shape in [(4,), (10, 4), (2, 5, 4)]:
    x = jnp.ones(shape)
    print(f"  {str(shape):<10} -> {apply_linear(params, x).shape}")

# The whole layer is an ordinary argument, so grad just works.
g = jax.grad(lambda p: jnp.sum(apply_linear(p, jnp.ones((2, 4)))))(params)
print("\ngrad leaves:", jax.tree.map(lambda a: a.shape, g))

no_bias = init_linear(jax.random.key(0), 4, 3, use_bias=False)
print("use_bias=False keys:", sorted(no_bias))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("linear_pure")

# hint("linear_pure")      # stuck? nudge without the answer
# solution("linear_pure")  # spoiler: the reference implementation
# status()                 # your dashboard across all problems